# 11 Stint Analysis

Stints connect tyre compound, tyre age, lap duration, and pit strategy. This notebook turns Silver stints into tyre degradation and strategy signals for Gold feature engineering.

In [1]:
from pathlib import Path
import sys
from datetime import datetime
import json

import pandas as pd
import numpy as np
import plotly.express as px

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

SHARED = ROOT / "eda" / "shared" / "scripts"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from config import CLEANED_DATA_PATH

NOTEBOOK_NAME = "11_stint_analysis"
OUTPUT_TABLES = ROOT / "eda" / "silver" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "silver" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "silver" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "silver" / "insights"
CHECKPOINTS = ROOT / "eda" / "silver" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(title: str, observations: list[str], issues: list[str], recommendations: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n"
    (INSIGHTS / f"{NOTEBOOK_NAME}.md").write_text(content, encoding="utf-8")

def add_event_type(sessions: pd.DataFrame) -> pd.DataFrame:
    sessions = sessions.copy()
    sessions["event_type"] = np.where(
        sessions["session_name"].astype(str).str.lower().eq("sprint"),
        "SPRINT_RACE",
        "GRAND_PRIX_RACE",
    )
    return sessions

def save_fig(fig, name: str):
    fig.write_html(OUTPUT_CHARTS / f"{name}.html", include_plotlyjs="cdn")
    fig.show()

print("=" * 72)
print(f"SILVER STRATEGY EDA - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Cleaned data path: {CLEANED_DATA_PATH}")
print("=" * 72)


SILVER STRATEGY EDA - 11_stint_analysis
Start time: 2026-06-02 02:14:28.269590
Cleaned data path: D:\F1_WinRate_Predictor\data\cleaned


In [2]:
stints = pd.read_parquet(CLEANED_DATA_PATH / "stints.parquet")
laps = pd.read_parquet(CLEANED_DATA_PATH / "laps.parquet")
drivers = pd.read_parquet(CLEANED_DATA_PATH / "drivers.parquet")
sessions = add_event_type(pd.read_parquet(CLEANED_DATA_PATH / "sessions.parquet"))
session_result = pd.read_parquet(CLEANED_DATA_PATH / "session_result.parquet")

for column in ["lap_start", "lap_end", "tyre_age_at_start", "stint_number", "driver_number", "session_key"]:
    if column in stints.columns:
        stints[column] = pd.to_numeric(stints[column], errors="coerce")
stints["stint_length"] = stints["lap_end"] - stints["lap_start"] + 1
stints = stints.merge(sessions[["session_key", "year", "event_type", "circuit_short_name"]], on="session_key", how="left")
stints = stints.merge(drivers[["session_key", "driver_number", "full_name", "team_name"]].drop_duplicates(["session_key", "driver_number"]), on=["session_key", "driver_number"], how="left")
stints["driver_id"] = stints["full_name"].fillna("Driver " + stints["driver_number"].astype(int).astype(str))

coverage = pd.DataFrame([{
    "stint_rows": len(stints),
    "sessions": stints["session_key"].nunique(),
    "drivers": stints["driver_number"].nunique(),
    "compounds": stints["compound"].nunique(),
}])
coverage.to_csv(OUTPUT_TABLES / "stint_coverage.csv", index=False)
display(coverage)

,stint_rows,sessions,drivers,compounds
0,3254,68,31,6


## 1. Stint Length and Compound Mix

Compound mix explains how teams trade pace against durability. Stint length by compound gives the first approximation of tyre life before modeling lap-level degradation.

In [3]:
stint_summary = stints.groupby("compound", as_index=False).agg(
    stints=("stint_number", "count"),
    avg_stint_length=("stint_length", "mean"),
    median_stint_length=("stint_length", "median"),
    p90_stint_length=("stint_length", lambda s: s.quantile(0.90)),
    avg_tyre_age_start=("tyre_age_at_start", "mean"),
).sort_values("stints", ascending=False)
stint_summary.to_csv(OUTPUT_TABLES / "compound_stint_summary.csv", index=False)
display(stint_summary)

fig = px.bar(
    stint_summary,
    x="compound",
    y="stints",
    color="median_stint_length",
    title="Tyre Compound Usage and Median Stint Length",
)
save_fig(fig, "compound_usage")

,compound,stints,avg_stint_length,median_stint_length,p90_stint_length,avg_tyre_age_start
2,MEDIUM,1451,18.461751,19.0,31.0,1.427981
0,HARD,1056,26.142045,26.0,42.5,0.939394
3,SOFT,462,13.582251,13.0,25.0,1.292208
1,INTERMEDIATE,271,13.298893,11.0,31.0,0.512915
4,UNKNOWN,7,10.000000,10.0,13.4,2.571429
5,WET,7,5.285714,4.0,9.0,1.428571


In [4]:
fig = px.box(
    stints.dropna(subset=["stint_length", "compound"]),
    x="compound",
    y="stint_length",
    color="compound",
    title="Stint Length Distribution by Compound",
)
save_fig(fig, "stint_length_by_compound")

## 2. Lap-Time Degradation by Compound

The lap table is joined to stint ranges to estimate tyre age within a stint. This is intentionally aggregated before plotting so the notebook remains light.

In [5]:
lap_work = laps[["session_key", "driver_number", "lap_number", "lap_duration", "is_pit_out_lap"]].copy()
lap_work["lap_number"] = pd.to_numeric(lap_work["lap_number"], errors="coerce")
lap_work["lap_duration"] = pd.to_numeric(lap_work["lap_duration"], errors="coerce")
lap_work = lap_work[lap_work["lap_duration"].between(50, 900)]

stint_ranges = stints[["session_key", "driver_number", "stint_number", "compound", "lap_start", "lap_end", "tyre_age_at_start"]].dropna(subset=["lap_start", "lap_end"])
lap_stints = lap_work.merge(stint_ranges, on=["session_key", "driver_number"], how="inner")
lap_stints = lap_stints[(lap_stints["lap_number"] >= lap_stints["lap_start"]) & (lap_stints["lap_number"] <= lap_stints["lap_end"])]
lap_stints["stint_lap_index"] = lap_stints["lap_number"] - lap_stints["lap_start"] + 1
lap_stints["estimated_tyre_age"] = lap_stints["tyre_age_at_start"] + lap_stints["stint_lap_index"] - 1
degradation = lap_stints.groupby(["compound", "stint_lap_index"], as_index=False).agg(
    median_lap_duration=("lap_duration", "median"),
    avg_lap_duration=("lap_duration", "mean"),
    laps=("lap_duration", "count"),
)
degradation = degradation[degradation["laps"] >= 20]
degradation.to_csv(OUTPUT_TABLES / "compound_lap_degradation.csv", index=False)
display(degradation.head(30))

fig = px.line(
    degradation,
    x="stint_lap_index",
    y="median_lap_duration",
    color="compound",
    markers=True,
    title="Tyre Degradation Curve by Compound",
)
save_fig(fig, "compound_degradation_curve")

,compound,stint_lap_index,median_lap_duration,avg_lap_duration,laps
0,HARD,1.0,113.0980,114.619312,1007
1,HARD,2.0,94.4550,97.928356,979
2,HARD,3.0,93.4550,95.332413,970
3,HARD,4.0,93.1620,94.306763,963
4,HARD,5.0,92.4755,93.688646,956
5,HARD,6.0,90.9920,92.041166,949
6,HARD,7.0,91.3510,92.027934,949
7,HARD,8.0,90.6270,91.478491,939
8,HARD,9.0,90.1370,90.702465,937
9,HARD,10.0,90.1100,90.365830,929


## 3. Driver and Team Stint Management

Long stints are not automatically good; they must be interpreted with lap-time stability. We summarize who extends stints while keeping lap-time variance controlled.

In [6]:
driver_stint = lap_stints.groupby(["driver_number", "compound", "session_key", "stint_number"], as_index=False).agg(
    stint_laps=("lap_number", "count"),
    median_lap=("lap_duration", "median"),
    lap_std=("lap_duration", "std"),
)
driver_stint = driver_stint.merge(stints[["session_key", "driver_number", "stint_number", "driver_id", "team_name"]], on=["session_key", "driver_number", "stint_number"], how="left")
driver_management = driver_stint.groupby("driver_id", as_index=False).agg(
    avg_stint_laps=("stint_laps", "mean"),
    max_stint_laps=("stint_laps", "max"),
    avg_lap_std=("lap_std", "mean"),
    stints=("stint_number", "count"),
)
driver_management = driver_management[driver_management["stints"] >= 10].sort_values(["avg_stint_laps", "avg_lap_std"], ascending=[False, True])
driver_management.to_csv(OUTPUT_TABLES / "driver_stint_management.csv", index=False)
display(driver_management.head(20))

fig = px.scatter(
    driver_management,
    x="avg_stint_laps",
    y="avg_lap_std",
    size="stints",
    hover_name="driver_id",
    title="Driver Stint Extension vs Lap-Time Variability",
)
save_fig(fig, "driver_stint_management")

,driver_id,avg_stint_laps,max_stint_laps,avg_lap_std,stints
1,Arvid LINDBLAD,25.300000,42,8.313428,10
4,Daniel RICCIARDO,22.083333,75,6.042000,48
15,Lando NORRIS,21.270968,77,8.124597,155
7,Franco COLAPINTO,21.077778,52,7.477238,90
3,Charles LECLERC,20.767296,77,7.849529,159
22,Oscar PIASTRI,20.762821,77,7.963227,156
5,Esteban OCON,20.740260,56,7.369909,154
9,George RUSSELL,20.723926,77,7.839470,163
11,Jack DOOHAN,20.600000,45,6.588180,15
19,Max VERSTAPPEN,20.521739,51,8.126974,161


In [7]:
team_strategy = stints.groupby(["session_key", "team_name", "driver_number"], as_index=False).agg(
    stint_count=("stint_number", "nunique"),
    compounds_used=("compound", "nunique"),
    max_stint_length=("stint_length", "max"),
)
team_strategy_summary = team_strategy.groupby("team_name", as_index=False).agg(
    avg_stints=("stint_count", "mean"),
    one_stop_rate=("stint_count", lambda s: (s == 2).mean()),
    two_stop_rate=("stint_count", lambda s: (s == 3).mean()),
    three_plus_stop_rate=("stint_count", lambda s: (s >= 4).mean()),
    avg_compounds_used=("compounds_used", "mean"),
)
team_strategy_summary.to_csv(OUTPUT_TABLES / "team_strategy_patterns.csv", index=False)
display(team_strategy_summary.sort_values("avg_stints", ascending=False))

fig = px.bar(
    team_strategy_summary.sort_values("avg_stints"),
    x="avg_stints",
    y="team_name",
    orientation="h",
    color="avg_compounds_used",
    title="Team Strategy Pattern: Average Stints and Compound Variety",
)
save_fig(fig, "team_strategy_patterns")

,team_name,avg_stints,one_stop_rate,two_stop_rate,three_plus_stop_rate,avg_compounds_used
6,Kick Sauber,2.500000,0.350000,0.358333,0.116667,1.900000
1,Aston Martin,2.492537,0.343284,0.335821,0.134328,1.880597
11,Red Bull Racing,2.477612,0.358209,0.313433,0.141791,1.873134
9,RB,2.448276,0.327586,0.362069,0.120690,1.965517
12,Williams,2.446154,0.361538,0.330769,0.115385,1.876923
5,Haas F1 Team,2.432836,0.402985,0.298507,0.119403,1.873134
8,Mercedes,2.411765,0.382353,0.323529,0.102941,1.838235
4,Ferrari,2.392593,0.407407,0.303704,0.103704,1.851852
7,McLaren,2.373134,0.350746,0.373134,0.074627,1.798507
0,Alpine,2.351145,0.419847,0.274809,0.114504,1.885496


## 4. Undercut Proxy and Used-Tyre Signal

A full undercut model needs exact pit and position timing. As a Silver proxy, we compare median lap time in the final laps before a stint change with the opening flying laps after the next stint begins.

In [8]:
ordered = lap_stints.sort_values(["session_key", "driver_number", "stint_number", "stint_lap_index"])
stint_edges = []
for (session_key, driver_number), group in ordered.groupby(["session_key", "driver_number"]):
    grouped = {stint: data for stint, data in group.groupby("stint_number")}
    for stint_number in sorted(grouped):
        next_number = stint_number + 1
        if next_number not in grouped:
            continue
        before = grouped[stint_number].tail(3)["lap_duration"].median()
        after = grouped[next_number].head(3)["lap_duration"].median()
        stint_edges.append({
            "session_key": session_key,
            "driver_number": driver_number,
            "stint_number": stint_number,
            "next_stint_number": next_number,
            "pre_stop_median_lap": before,
            "post_stop_median_lap": after,
            "lap_delta_after_stop": after - before,
        })
undercut_proxy = pd.DataFrame(stint_edges)
undercut_proxy.to_csv(OUTPUT_TABLES / "undercut_proxy_lap_delta.csv", index=False)
display(undercut_proxy.describe().reset_index() if len(undercut_proxy) else undercut_proxy)

fig = px.histogram(
    undercut_proxy.dropna(subset=["lap_delta_after_stop"]),
    x="lap_delta_after_stop",
    nbins=60,
    title="Post-Stop Lap Delta Proxy",
)
save_fig(fig, "undercut_proxy_lap_delta")

,index,session_key,driver_number,stint_number,next_stint_number,pre_stop_median_lap,post_stop_median_lap,lap_delta_after_stop
0,count,1869.000000,1869.000000,1869.000000,1869.000000,1869.000000,1869.000000,1869.000000
1,mean,9860.239165,29.108079,1.624933,2.624933,99.174604,101.575227,2.400623
2,std,441.481789,24.179116,0.891933,0.891933,24.826918,24.206741,19.900302
3,min,9472.000000,1.000000,1.000000,2.000000,69.887000,67.694000,-95.334000
4,25%,9574.000000,11.000000,1.000000,2.000000,83.151000,83.290000,-2.182000
5,50%,9839.000000,22.000000,1.000000,2.000000,94.182000,96.338000,-1.088000
6,75%,9963.000000,44.000000,2.000000,3.000000,102.278000,110.380500,4.531000
7,max,11291.000000,87.000000,6.000000,7.000000,219.048000,177.756000,92.043000


In [9]:
stints["tyre_start_class"] = np.where(stints["tyre_age_at_start"].fillna(0) <= 1, "New/near-new", "Used")
used_tyre_summary = stints.groupby(["compound", "tyre_start_class"], as_index=False).agg(
    stints=("stint_number", "count"),
    avg_stint_length=("stint_length", "mean"),
    median_stint_length=("stint_length", "median"),
)
used_tyre_summary.to_csv(OUTPUT_TABLES / "used_tyre_strategy.csv", index=False)
display(used_tyre_summary)

fig = px.bar(
    used_tyre_summary,
    x="compound",
    y="median_stint_length",
    color="tyre_start_class",
    barmode="group",
    title="New vs Used Tyre Stint Length",
)
save_fig(fig, "used_tyre_strategy")

,compound,tyre_start_class,stints,avg_stint_length,median_stint_length
0,HARD,New/near-new,1002,26.742515,27.0
1,HARD,Used,54,15.000000,17.0
2,INTERMEDIATE,New/near-new,224,12.848214,11.0
3,INTERMEDIATE,Used,47,15.446809,5.0
4,MEDIUM,New/near-new,1142,18.991243,19.0
5,MEDIUM,Used,309,16.504854,19.0
6,SOFT,New/near-new,295,13.023729,12.0
7,SOFT,Used,167,14.568862,14.0
8,UNKNOWN,Used,7,10.000000,10.0
9,WET,New/near-new,5,5.800000,4.0


## Final Stint Report

In [10]:
top_compound = stint_summary.iloc[0].to_dict() if len(stint_summary) else {}
top_manager = driver_management.iloc[0].to_dict() if len(driver_management) else {}
report = {
    "notebook": NOTEBOOK_NAME,
    "timestamp": datetime.now().isoformat(),
    "status": "PASS",
    "stint_rows": int(len(stints)),
    "lap_stint_rows": int(len(lap_stints)),
    "most_used_compound": top_compound.get("compound"),
    "most_used_compound_stints": int(top_compound.get("stints", 0)) if top_compound else 0,
    "top_stint_extension_driver": top_manager.get("driver_id"),
    "top_driver_avg_stint_laps": float(top_manager.get("avg_stint_laps", 0)) if top_manager else 0.0,
}
write_report("stint_analysis", report)
write_insight(
    "Silver Stint Analysis Insights",
    [
        f"Analyzed {report['stint_rows']:,} stint records and {report['lap_stint_rows']:,} lap-stint joins.",
        f"Most used compound: {report['most_used_compound']}.",
        f"Top stint extension profile: {report['top_stint_extension_driver']}.",
    ],
    [],
    [
        "Use compound, stint_lap_index, estimated_tyre_age, and stint_count as Gold tyre strategy features.",
        "Model degradation by compound separately; tyre behavior is not homogeneous across compounds.",
        "Treat undercut_proxy as exploratory until pit and position timing are modeled directly.",
    ],
)
(CHECKPOINTS / "silver_stint_analysis_completed.txt").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(report)

{'notebook': '11_stint_analysis', 'timestamp': '2026-06-02T02:14:31.010490', 'status': 'PASS', 'stint_rows': 3254, 'lap_stint_rows': 63684, 'most_used_compound': 'MEDIUM', 'most_used_compound_stints': 1451, 'top_stint_extension_driver': 'Arvid LINDBLAD', 'top_driver_avg_stint_laps': 25.3}
